In [3]:
import pandas as pd
import numpy as np
import os
import gc

# 1. Cấu hình đường dẫn
RAW_DIR = '../dataset/raw/'
PROCESSED_DIR = '../dataset/processed/'

print("1. Đang load Master Data (đã ép cân)...")
df = pd.read_parquet(os.path.join(PROCESSED_DIR, 'master_data.parquet'))

print("2. Đang load dữ liệu Giá (sell_prices.csv)...")
prices = pd.read_csv(os.path.join(RAW_DIR, 'sell_prices.csv'))

print("3. Đang hợp nhất Giá vào dữ liệu chính...")
# Merge giá dựa trên cửa hàng, mã hàng và tuần
df = pd.merge(df, prices, on=['store_id', 'item_id', 'wm_yr_wk'], how='left')
del prices
gc.collect()

print("4. Đang tạo Đặc trưng Giá (Price Features)...")
df['price_max'] = df.groupby(['store_id', 'item_id'])['sell_price'].transform('max')
df['price_discount'] = df['sell_price'] / df['price_max']
df['price_momentum'] = df['sell_price'] / df.groupby(['store_id', 'item_id'])['sell_price'].shift(1)

print("5. Đang xử lý Missing Value & Sự kiện (Calendar Features)...")
# Xử lý NaN thành 'No_Event' để không bị mất dòng khi đưa vào mô hình
event_cols = ['event_name_1', 'event_type_1', 'event_name_2', 'event_type_2']
for col in event_cols:
    if df[col].dtype.name == 'category':
        df[col] = df[col].cat.add_categories(['No_Event'])
    df[col] = df[col].fillna('No_Event')

# Mã hóa (Label Encode) các cột chữ thành số
cols_to_encode = ['item_id', 'dept_id', 'cat_id', 'store_id', 'state_id'] + event_cols
for col in cols_to_encode:
    df[col] = df[col].astype('category').cat.codes

print("6. Đang tạo Đặc trưng Thời gian (Time Features)...")
df['day'] = df['date'].dt.day
df['is_weekend'] = df['wday'].isin([1, 2]).astype(np.int8)

print("7. Đang lưu tập dữ liệu Đã trích xuất đặc trưng...")
# Nới lỏng ra float32 để màn hình in ra đẹp đẽ, không bị la ó nha 🎀
cols_float = ['sell_price', 'price_max', 'price_discount', 'price_momentum']
for col in cols_float:
    df[col] = df[col].astype(np.float32)

# Lưu thành file Parquet chuẩn hóa
featured_file = os.path.join(PROCESSED_DIR, 'featured_data.parquet')
df.to_parquet(featured_file, index=False)

print(f"Dữ liệu tại: {featured_file}")
display(df[['date', 'item_id', 'demand', 'sell_price', 'price_discount', 'event_name_1']].head())

1. Đang load Master Data (đã ép cân)...
2. Đang load dữ liệu Giá (sell_prices.csv)...
3. Đang hợp nhất Giá vào dữ liệu chính...
4. Đang tạo Đặc trưng Giá (Price Features)...
5. Đang xử lý Missing Value & Sự kiện (Calendar Features)...
6. Đang tạo Đặc trưng Thời gian (Time Features)...
7. Đang lưu tập dữ liệu Đã trích xuất đặc trưng...
Dữ liệu tại: ../dataset/processed/featured_data.parquet


,date,item_id,demand,sell_price,price_discount,event_name_1
0,2011-01-29,1437,0,NaN,NaN,30
1,2011-01-29,1438,0,NaN,NaN,30
2,2011-01-29,1439,0,NaN,NaN,30
3,2011-01-29,1440,0,NaN,NaN,30
4,2011-01-29,1441,0,NaN,NaN,30
